# Strengthening the SAE KG‑Repair Spine — Demo

**Artifact:** *Strengthening the SAE KG‑Repair Spine: Non‑Eval‑Aligned Controls + Downstream Test* (iter‑9, experiment 3 / reviewer R5).

Single SAE latents are unreliable units of analysis (feature absorption / splitting / non‑atomicity).
The parent project builds **cluster‑/group‑level units** ("KG absorbers": one auditable extra latent that
covers a parent concept's silent *sub‑context hole* X). The original headline compared the KG‑named
absorber against only a **weak single‑random‑latent control** — a near‑definitional test. This experiment
**strengthens** that spine with:

1. **Stronger, non‑eval‑aligned controls** (none ranked by per‑sub‑context precision):
   `dense_jtt` / `dense_dom` (decoder‑projection argmax of a dense probe — turns out to be the *parent*),
   label‑free `S_mag` (magnitude) and `S_rec` (recall), and `*_poolX` (same eligibility pool, different
   ranking). For each *hole × control*: paired bootstrap (B=10,000) → one‑sided p → an **augmented
   Benjamini–Hochberg FDR** over the whole family (cross‑checked vs `statsmodels`).
2. A **downstream‑capability** test: held‑out recall of *(parent + KG absorber)* vs *(parent + dense
   logistic probe)*, paired bootstrap.

**What this notebook does.** The heavy part of the original run (Gemma‑Scope SAE encodings, per‑window
bootstraps) is **$0 cached** and produced the per‑comparison statistics. This demo loads those statistics
and **re‑runs the analysis layer verbatim** — the augmented BH FDR, the downstream per‑hole bootstrap, and
the verdict fork — reproducing the published headline:

> `REPAIR_IS_NON_TAUTOLOGICAL_LOCALIZATION + DOWNSTREAM_CAPABILITY_NULL_TEMPER`

It runs in well under a second, on CPU, with no GPU and no model forward pass.

In [ ]:
# --- install dependencies (Colab-aware) ---
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# This analysis uses ONLY packages pre-installed on Colab (numpy, scipy, pandas, matplotlib, statsmodels).
# On Colab: skip them (installing would corrupt the pre-loaded C extensions).
# Locally: install at Colab's exact versions so the local env mirrors Colab.
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'pandas==2.2.2',
         'matplotlib==3.10.0', 'statsmodels==0.14.6')

In [ ]:
# --- imports (from the original method.py + matplotlib for the demo viz) ---
import os, sys, json
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

In [ ]:
# --- data loading (GitHub raw URL with local fallback for Colab) ---
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7ee30c-catching-silent-feature-absorption-in-fr/main/round-9/experiment-3/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("loaded keys:", list(data.keys()))
print("gating check  :", {k: data["gating_check"][k] for k in ("pass", "cosine", "L0")})
print("control rows  :", len(data["stronger_control_table"]),
      "| holes:", sum(r["is_hole"] for r in data["stronger_control_table"]))
print("downstream    :", list(data["downstream_capability"].keys()))

## Configuration

All tunable parameters live here. These are the **original full‑scale values** from `core.py`
(`SEED=1234`, `B_BOOT=10000`, `FDR_ALPHA=0.05`) — the analysis layer is sub‑second, so no scaling‑down is
needed to fit the demo budget. Smaller demo alternatives are given in comments. `N_MAX_ROWS=None` analyzes
all 63 *hole × control* rows (set e.g. `10` for a smaller, faster slice).

In [ ]:
# ---- tunable parameters (original full-scale values; analysis runs in < 1 s) ----
SEED       = 1234     # RNG seed for the paired bootstrap            (core.SEED)
B_BOOT     = 10000    # paired-bootstrap resamples                   (core.B_BOOT)   ; demo-min alt: 200
FDR_ALPHA  = 0.05     # Benjamini-Hochberg target FDR                (core.FDR_ALPHA)
N_MIN_EVAL = 30       # min held-out windows -> non-"descriptive"    (core.N_MIN_EVAL)
N_MAX_ROWS = None     # analyze ALL hole x control rows; set e.g. 10 for a smaller demo

rng = np.random.default_rng(SEED)   # core.py reseeds its global rng exactly this way
print(f"SEED={SEED}  B_BOOT={B_BOOT}  FDR_ALPHA={FDR_ALPHA}  N_MAX_ROWS={N_MAX_ROWS}")

## 1. Statistical primitives (verbatim from `core.py`)

The two inference primitives the analysis is built on. `paired_bootstrap_diff` is the paired bootstrap of a
mean difference (95% CI + one‑sided p); `benjamini_hochberg` is the hand‑rolled BH FDR that the experiment
cross‑checks against `statsmodels`. Copied unchanged from the original `core.py`.

In [ ]:
# ---- core.py VERBATIM ----
def paired_bootstrap_diff(diff_per_item, B=B_BOOT):
    """Paired bootstrap of mean(diff). Returns 95% CI + one-sided p (H0: mean<=0)."""
    d = np.asarray(diff_per_item, dtype=np.float64)
    n = len(d)
    if n == 0:
        return {"diff": 0.0, "ci_lo": 0.0, "ci_hi": 0.0, "excl_0": False, "n": 0, "p_one_sided": 1.0}
    idx = rng.integers(0, n, size=(B, n))
    bs = d[idx].mean(1)
    lo, hi = np.percentile(bs, [2.5, 97.5])
    p_one = (1.0 + float((bs <= 0).sum())) / (B + 1.0)
    return {"diff": float(d.mean()), "ci_lo": float(lo), "ci_hi": float(hi),
            "excl_0": bool(lo > 0 or hi < 0), "n": int(n), "p_one_sided": float(p_one)}


def benjamini_hochberg(pvals, alpha=FDR_ALPHA):
    """BH FDR-adjusted q-values + count surviving. Hand-rolled (matches statsmodels fdr_bh)."""
    p = np.asarray(pvals, dtype=np.float64)
    n = len(p)
    if n == 0:
        return np.array([]), 0
    order = np.argsort(p)
    ranked = p[order]
    q = ranked * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0.0, 1.0)
    out = np.empty(n)
    out[order] = q
    return out, int((out <= alpha).sum())

## 2. Load the per‑comparison table

`stronger_control_table` holds one row per *(parent hole X × control)* comparison. Each row carries the
KG‑named absorber, the candidate pool, the recall `gains`, and `kg_minus_control` — the paired‑bootstrap
KG‑minus‑control difference with its CI and one‑sided p (these are the **cached outputs** of the expensive
per‑window bootstrap). `downstream` holds the per‑sub‑context recall of the repaired unit vs the dense
probe.

In [ ]:
table = data["stronger_control_table"]
if N_MAX_ROWS is not None:
    table = table[:N_MAX_ROWS]
downstream = data["downstream_capability"]

n_holes = sum(r["is_hole"] for r in table)
fams = sorted(set(r["family"] for r in table))
print(f"{len(table)} (hole x control) rows; {n_holes} are parent 'holes'; families = {fams}")
# peek at one homograph-taxonomic hole
ex = next(r for r in table if r["family"] == "homograph_taxonomic" and r["is_hole"])
print(f"\nexample hole: concept={ex['concept']} X={ex['X']!r} kg_absorber={ex['kg_absorber']} "
      f"gain_kg={ex['gains']['kg']:.3f}")
print("  kg-minus-control (dense_jtt):", {k: round(ex['kg_minus_control']['dense_jtt'][k], 3)
      for k in ('diff', 'ci_lo', 'ci_hi', 'p_one_sided')})

## 3. Augmented Benjamini–Hochberg FDR  (verbatim from `method.py`)

Collect the one‑sided p over every *hole × stronger‑control* comparison (singleton‑pool ties are structural
and excluded), apply BH FDR, write `bh_q` / `survives_FDR` back into each row, and cross‑check the q‑values
against `statsmodels`. This is `apply_augmented_bh` copied unchanged.

In [ ]:
# ---- method.py VERBATIM ----
def apply_augmented_bh(all_rows, restrict_to_holes=True):
    """Collect one-sided p across (concept,X,stronger-control) over the HOLE family and apply BH FDR;
    cross-check vs statsmodels. Writes bh_q / survives_FDR back into each kg_minus_control entry."""
    NAMED = ["dense_jtt", "dense_dom", "S_mag_global", "S_rec_global", "S_mag_poolX", "S_rec_poolX"]
    refs = []   # (control, family, dict)
    for r in all_rows:
        if restrict_to_holes and not r["is_hole"]:
            continue
        for cname in NAMED:
            ent = r["kg_minus_control"].get(cname)
            if not ent or "p_one_sided" not in ent:
                continue
            if ent.get("tie_by_pool_singleton"):
                continue   # singleton-pool ties are structural, not inferential -> excluded from FDR
            refs.append((cname, r["family"], ent))
    pvals = [e["p_one_sided"] for (_, _, e) in refs]
    q, n_sig = benjamini_hochberg(pvals, FDR_ALPHA)
    sm_ok = None
    try:
        from statsmodels.stats.multitest import multipletests
        if pvals:
            _, q_sm, _, _ = multipletests(pvals, alpha=FDR_ALPHA, method="fdr_bh")
            sm_ok = bool(np.allclose(q_sm, q, atol=1e-9))
    except Exception:  # noqa: BLE001
        sm_ok = None
    per_control = defaultdict(lambda: {"tested": 0, "survive": 0})
    per_family = defaultdict(lambda: {"tested": 0, "survive": 0})
    per_control_family = defaultdict(lambda: defaultdict(lambda: {"tested": 0, "survive": 0}))
    for i, (cname, fam, ent) in enumerate(refs):
        ent["bh_q"] = float(q[i])
        surv = bool(q[i] <= FDR_ALPHA and ent["diff"] > 0)
        ent["survives_FDR"] = surv
        per_control[cname]["tested"] += 1; per_control[cname]["survive"] += int(surv)
        per_family[fam]["tested"] += 1; per_family[fam]["survive"] += int(surv)
        per_control_family[cname][fam]["tested"] += 1
        per_control_family[cname][fam]["survive"] += int(surv)
    return {
        "method": "Benjamini-Hochberg FDR", "alpha": FDR_ALPHA, "family": "holes x stronger-controls",
        "n_comparisons": len(refs), "n_survive_total": int(n_sig),
        "statsmodels_crosscheck_matches": sm_ok,
        "per_control_survive": {k: dict(v) for k, v in per_control.items()},
        "per_family_survive": {k: dict(v) for k, v in per_family.items()},
        "per_control_per_family": {k: {f: dict(vv) for f, vv in v.items()} for k, v in per_control_family.items()},
    }


augmented = apply_augmented_bh(table, restrict_to_holes=True)
print(f"augmented BH: {augmented['n_comparisons']} hole-vs-control comparisons, "
      f"{augmented['n_survive_total']} survive FDR<={FDR_ALPHA}")
print("statsmodels q-value cross-check matches:", augmented["statsmodels_crosscheck_matches"])
print("per-family survive:", augmented["per_family_survive"])

# reproduction check vs the published full run
if N_MAX_ROWS is None:
    pub = data["published_augmented_multiplicity"]
    print(f"\nPUBLISHED: {pub['n_comparisons']} comparisons, {pub['n_survive_total']} survive  "
          f"-> reproduced exactly: {augmented['n_survive_total'] == pub['n_survive_total']}")

## 4. Downstream per‑hole bootstrap (re‑run on cached recalls)

For each concept we recompute the **per‑hole** paired bootstrap of *(recall of repaired unit − recall of
parent+dense‑probe)* from the cached per‑sub‑context recalls, exactly as `downstream_capability` does
(`hole_diffs = [v["diff_A_minus_B"] for v in per_subcontext]`). Because we reuse the same `SEED` and
`B_BOOT`, this reproduces the published bootstrap to the digit. Negative diffs ⇒ the dense probe out‑recalls
the repaired unit (the basis of the `NULL_TEMPER` half of the verdict).

In [ ]:
recomputed_downstream = {}
print(f"{'concept':<10} {'n_holes':>7} {'diff(A-B)':>10} {'CI':>22} {'excl0':>6}   published")
for c, dd in downstream.items():
    px = dd.get("per_subcontext", {})
    diffs = np.array([v["diff_A_minus_B"] for v in px.values()])
    if len(diffs) == 0:
        continue
    bs = paired_bootstrap_diff(diffs, B=B_BOOT)        # core.py primitive, real bootstrap
    recomputed_downstream[c] = bs
    pubb = dd.get("per_hole_diff_bootstrap", {})
    match = (abs(bs["diff"] - pubb.get("diff", 9)) < 1e-9) and (bs["excl_0"] == pubb.get("excl_0"))
    ci = f"[{bs['ci_lo']:+.3f}, {bs['ci_hi']:+.3f}]"
    print(f"{c:<10} {bs['n']:>7} {bs['diff']:>+10.4f} {ci:>22} {str(bs['excl_0']):>6}   "
          f"match={match}")

## 5. Verdict fork (verbatim from `method.py`)

`build_verdict` turns the FDR‑annotated rows + the downstream bootstraps into the two‑part verdict:
`repair_verdict` (is the KG repair non‑tautological vs the stronger controls?) and `capability_verdict`
(does the repaired unit out‑recall a strong dense probe downstream?). Copied unchanged.

In [ ]:
# ---- method.py VERBATIM ----
def build_verdict(all_rows, augmented, downstream_all):
    NAMED_CONCEPT = ["dense_jtt", "dense_dom", "S_mag_global", "S_rec_global"]
    hole_rows = [r for r in all_rows if r["is_hole"]]
    spelling_tax = [r for r in hole_rows if r["family"] in ("spelling", "homograph_taxonomic")]
    # how many spelling+tax holes have KG beat ALL named concept-level controls at FDR<=0.05
    n_beat_all_named = 0
    n_beat_all_named_incl_pool = 0
    for r in spelling_tax:
        ok_named = True
        for c in NAMED_CONCEPT:
            ent = r["kg_minus_control"].get(c, {})
            if ent.get("status") == "control_undefined":
                continue
            if not (ent.get("survives_FDR") and ent.get("diff", 0) > 0):
                ok_named = False
        if ok_named:
            n_beat_all_named += 1
        pool_ok = True
        for c in ("S_mag_poolX", "S_rec_poolX"):
            ent = r["kg_minus_control"].get(c, {})
            if ent.get("status") == "control_undefined" or ent.get("tie_by_pool_singleton"):
                continue
            if r["pool_size_X"] > 1 and not (ent.get("excl_0") and ent.get("diff", 0) > 0):
                pool_ok = False
        if ok_named and pool_ok:
            n_beat_all_named_incl_pool += 1
    n_spelling_tax_holes = len(spelling_tax)
    majority = n_beat_all_named >= max(1, (n_spelling_tax_holes + 1) // 2)

    # per-family breakdown (spelling+tax = headline; numeric = supplementary, expected mixed since the
    # stronger controls are genuinely non-trivial there -> demonstrates the controls are not strawmen)
    per_family = {}
    for fam in ("spelling", "homograph_taxonomic", "numeric"):
        fam_rows = [r for r in hole_rows if r["family"] == fam]
        nbeat = 0
        for r in fam_rows:
            ok = True
            for c in NAMED_CONCEPT:
                ent = r["kg_minus_control"].get(c, {})
                if ent.get("status") == "control_undefined":
                    continue
                if not (ent.get("survives_FDR") and ent.get("diff", 0) > 0):
                    ok = False
            nbeat += int(ok)
        # holes where some stronger control MATCHES OR BEATS the KG (control wins / ties) -> control is non-trivial
        n_control_competitive = 0
        for r in fam_rows:
            comp = any(r["kg_minus_control"].get(c, {}).get("diff", 1.0) <= 0
                       for c in NAMED_CONCEPT if r["kg_minus_control"].get(c, {}).get("status") != "control_undefined")
            n_control_competitive += int(comp)
        per_family[fam] = {"n_holes": len(fam_rows), "n_kg_beats_all_named_FDR": nbeat,
                           "n_holes_a_stronger_control_matches_or_beats_kg": n_control_competitive}

    if majority:
        repair_verdict = "REPAIR_IS_NON_TAUTOLOGICAL_LOCALIZATION"
        # precision-specific refinement on holes with |pool|>1
        multi = [r for r in spelling_tax if r["pool_size_X"] > 1]
        beat_pool = 0
        for r in multi:
            okp = True
            for c in ("S_mag_poolX", "S_rec_poolX"):
                ent = r["kg_minus_control"].get(c, {})
                if ent.get("status") == "control_undefined" or ent.get("tie_by_pool_singleton"):
                    continue
                if not (ent.get("survives_FDR") and ent.get("diff", 0) > 0):
                    okp = False
            beat_pool += int(okp)
        precision_specific = bool(multi and beat_pool >= max(1, (len(multi) + 1) // 2))
    else:
        repair_verdict = "REPAIR_SELF_CONSISTENT_TEMPER"
        precision_specific = False

    # capability verdict: FOUND only if the repaired unit out-recalls the strong dense probe on the
    # better-powered pooled-window bootstrap (diff>0, CI excl 0) for some concept.
    found = False
    for concept, dd in downstream_all.items():
        pooled = dd.get("pooled_window_diff_bootstrap")
        ph = dd.get("per_hole_diff_bootstrap")
        if pooled and pooled.get("excl_0") and pooled.get("diff", 0) > 0:
            found = True
        if ph and ph.get("excl_0") and ph.get("diff", 0) > 0:
            found = True
    capability_verdict = "DOWNSTREAM_CAPABILITY_FOUND" if found else "DOWNSTREAM_CAPABILITY_NULL_TEMPER"
    temper_language = (
        "The repaired unit does not out-recall a strong dense probe on held-out classification; the "
        "repair demonstrates correct LOCALIZATION (an auditable, per-sub-context handle the single dense "
        "hyperplane lacks -- the dense decoder-projection does not localize to any single SAE latent, "
        "argmax_is_anchor), NOT downstream recall utility. We temper all 'measured repair utility' "
        "language to 'measured, localized recall RECOVERY vs non-eval-aligned controls'."
    )
    numeric_note = (
        "Numeric is SUPPLEMENTARY and HONESTLY MIXED: on several numeric sub-contexts (e.g. integer, "
        "year, currency, comma_number) a stronger non-eval-aligned control matches or out-recovers the "
        "KG-named absorber -- demonstrating the controls are genuinely non-trivial (not strawmen) and "
        "that numeric localization is weaker than spelling/taxonomic; only ordinal/date/decimal show "
        "clean KG wins over the dense controls."
    )
    return {
        "repair_verdict": repair_verdict,
        "precision_specific": precision_specific,
        "n_spelling_tax_holes": n_spelling_tax_holes,
        "n_holes_kg_beats_all_named_controls_FDR": n_beat_all_named,
        "n_holes_kg_beats_named_plus_poolmatched": n_beat_all_named_incl_pool,
        "per_family": per_family,
        "capability_verdict": capability_verdict,
        "overall_verdict": f"{repair_verdict} + {capability_verdict}",
        "temper_language": temper_language if capability_verdict == "DOWNSTREAM_CAPABILITY_NULL_TEMPER" else None,
        "numeric_supplementary_note": numeric_note,
        "named_controls": NAMED_CONCEPT,
    }


verdict = build_verdict(table, augmented, downstream)
print("OVERALL VERDICT:", verdict["overall_verdict"])
print(f"\nrepair: KG beats ALL named controls on "
      f"{verdict['n_holes_kg_beats_all_named_controls_FDR']}/{verdict['n_spelling_tax_holes']} "
      f"spelling+taxonomic holes (precision_specific={verdict['precision_specific']})")
for fam, fv in verdict["per_family"].items():
    print(f"  {fam:<20} {fv['n_kg_beats_all_named_FDR']}/{fv['n_holes']} clean KG wins  "
          f"({fv['n_holes_a_stronger_control_matches_or_beats_kg']} holes a control matches/beats KG)")
print("\ncapability:", verdict["capability_verdict"])
print("temper language:", verdict["temper_language"][:160], "...")

## 6. Results

**Left** — per‑family clean KG wins (KG beats *all four* named non‑eval‑aligned controls at FDR≤0.05):
homograph‑taxonomic is 3/3 clean, spelling is the bulk, numeric is honestly mixed (the controls are
non‑trivial there). **Right** — downstream per‑hole recall difference *(repaired unit − dense probe)* with
95% CI per concept: mostly ≤ 0, i.e. the repaired unit does **not** out‑recall a strong dense probe — the
demonstrated value is auditable *localization*, not downstream recall (`NULL_TEMPER`).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))

# ---- (left) per-family FDR survival ----
fams = ["homograph_taxonomic", "spelling", "numeric"]
n_hole = [verdict["per_family"][f]["n_holes"] for f in fams]
n_win  = [verdict["per_family"][f]["n_kg_beats_all_named_FDR"] for f in fams]
x = np.arange(len(fams)); w = 0.38
ax1.bar(x - w/2, n_hole, w, label="# holes", color="#9ecae1")
ax1.bar(x + w/2, n_win,  w, label="KG beats ALL named controls (FDR<=0.05)", color="#08519c")
for i, (h, win) in enumerate(zip(n_hole, n_win)):
    ax1.text(i - w/2, h + 0.2, str(h), ha="center", fontsize=9)
    ax1.text(i + w/2, win + 0.2, str(win), ha="center", fontsize=9)
ax1.set_xticks(x); ax1.set_xticklabels([f.replace("homograph_taxonomic", "homograph-tax") for f in fams])
ax1.set_ylabel("# parent holes"); ax1.set_title("Repair is non-tautological\n(KG vs stronger controls, by family)")
ax1.legend(fontsize=8, loc="upper right")

# ---- (right) downstream per-hole recall diff (repaired unit - dense probe) ----
cs = list(recomputed_downstream.keys())
diffs = [recomputed_downstream[c]["diff"] for c in cs]
los   = [recomputed_downstream[c]["diff"] - recomputed_downstream[c]["ci_lo"] for c in cs]
his   = [recomputed_downstream[c]["ci_hi"] - recomputed_downstream[c]["diff"] for c in cs]
colors = ["#de2d26" if d < 0 else "#31a354" for d in diffs]
ax2.bar(np.arange(len(cs)), diffs, yerr=[los, his], capsize=4, color=colors, alpha=0.85)
ax2.axhline(0, color="k", lw=0.8)
ax2.set_xticks(np.arange(len(cs))); ax2.set_xticklabels(cs)
ax2.set_ylabel("recall(repaired unit) - recall(parent+dense)")
ax2.set_title("Downstream NULL_TEMPER\n(per-hole bootstrap, 95% CI)")

plt.tight_layout()
plt.savefig("demo_results.png", dpi=110)
plt.show()
print("saved demo_results.png")

# ---- headline table: the 3 clean homograph-taxonomic holes ----
print("\nHomograph-taxonomic holes (KG-minus-control diff; * = survives FDR):")
hdr = f"{'X':<16}{'gain_kg':>8}" + "".join(f"{c:>14}" for c in ["dense_jtt", "dense_dom", "S_mag_global", "S_rec_global"])
print(hdr)
for r in table:
    if r["family"] == "homograph_taxonomic" and r["is_hole"]:
        line = f"{r['X']:<16}{r['gains']['kg']:>8.3f}"
        for c in ["dense_jtt", "dense_dom", "S_mag_global", "S_rec_global"]:
            e = r["kg_minus_control"].get(c, {})
            star = "*" if e.get("survives_FDR") else " "
            line += f"{e.get('diff', float('nan')):>13.3f}{star}"
        print(line)